# Workflows: Run a GWAS on Every Chromosome at Once with dsub

**Optional Extras class.** In **4.2** we ran a GWAS on **one region** of one chromosome, inside our app. A whole-genome analysis is **22 or more** such jobs, each with large files. Run one after another in the app, that takes hours, and the app must stay open the whole time.

**Workflow tools** solve this. They send each job to **its own cloud machine**, run them **at the same time**, save the results in your bucket, and shut the machines down. Your app only launches the jobs and collects the results.

Today we use **dsub**, the simplest of the three workflow tools in the Workbench, to run the 4.2 GWAS on **all 22 chromosomes in parallel**.

### By the end of this notebook you'll be able to

1. Say **when** to use a workflow tool instead of running code in your app
2. Describe the parts of a dsub job: **image**, **inputs**, **outputs**, **script**, and **logging**
3. Test a script **locally** first, then run it as **one** job, then as **many** (`--tasks`)
4. **Monitor** jobs with `dstat`, cancel them with `ddel`, and read their **logs** to fix errors
5. Compare **dsub**, **Cromwell** (WDL), and **Nextflow**, and know where to start with each

### 🔎 Analysis at a glance

| | |
|---|---|
| ❓ **Question** | Which variants in the *All of Us* **ClinVar** files are linked to type 2 diabetes? (The real question: **how** do you run one analysis on every chromosome?) |
| 🧪 **The analysis** | The 4.2 GWAS (QC filters, then `plink2 --glm`, adjusted for age, sex, and 10 PCs), on each chromosome **in its own dsub job** |
| 🎯 **Why** | 22 jobs run in about the time of one; the app doesn't have to do the work, or even stay on |
| ✅ **What we conclude** | That the workflow gives the **same results** as the app, for all 22 chromosomes |
| ⚠️ **What it can't tell us** | Much about diabetes: the ClinVar files hold variants already studied for disease, and few are common. For a real GWAS, use the **ACAF** files (Part 7) |
| 🔗 **Connections** | The sample, phenotype, and covariates from **4.1** and **4.2**; the data folders from *Where the omics data lives* |

| | |
|---|---|
| 📂 **Uses** | `gwas_samples.csv` (from **4.1**, or rebuilds it); the v9 **ClinVar** PLINK files; `gwas_one_chr.sh` (next to this notebook) |
| 💾 **Saves** | in your bucket's `data/workflows/`: logs and PLINK results for each chromosome; `gwas_clinvar_all.csv`, the combined results |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts: what a workflow tool does | 20 |
| 2. Setup: settings, files, and the bucket | 10 |
| 3. Test locally: one chromosome in the app | 10 |
| 4. One dsub job: the same chromosome in the cloud | 15 |
| 5. All 22 chromosomes at once | 20 |
| 6. Combine the results | 10 |
| 7. Monitor, troubleshoot, scale up, and clean up | 10 |
| 8. Cromwell and Nextflow | 10 |
| Wrap-up | 5 |

### ⏱️ Time and 💰 cost

| | |
|---|---|
| ⏱️ **Class time** | 110 minutes |
| ⏱️ **Run time** | about **30 minutes**, mostly waiting for jobs (Parts 4 and 5 take 10 to 15 minutes each) |
| 💰 **Data cost** | none, unless `gwas_samples.csv` is missing: then the cohort query (cents to under $1) |
| 💰 **Job cost** | under **$1**: 23 small cloud machines (`n2-standard-2`, about $0.10 an hour each) for 10 to 15 minutes each, plus their disks |
| 💰 **App cost** | about **$0.27 an hour** while the app is running. The app's disk costs about **$0.70 a day**, even while the app is stopped |
| **Estimated total** | about **$1** per student for this class |

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts: what a workflow tool does

### Two ways to run the same analysis

| | In your app (4.2) | With a workflow tool (today) |
|---|---|---|
| **Where it runs** | on the app's machine | on **new cloud machines**, one per job |
| **How many at once** | one after another | **many at once** (in parallel) |
| **Machine size** | the app's size | **chosen per job**: big jobs get big machines |
| **While it runs** | the app must stay on; the notebook is busy | the app can do other things, or even be **stopped** |
| **Best for** | exploring, small data, plots | the **same steps on many files**: chromosomes, samples, batches |

👉 **Rule of thumb:** develop and test in your app; once a step works and has to run **many times** or on **big** files, move it to a workflow.

### Is a workflow cheaper?

Often **faster**; **cheaper** only sometimes. The total computing is about the same: 22 machines for 15 minutes cost about as much as 1 machine for 5½ hours. What changes is **when** you pay, and for **what size** of machine.

| A workflow **saves** money when… | A workflow **costs more** when… |
|---|---|
| the alternative is a **big app** running for hours (including while you wait, or forget to stop it) | the job is **small**: each machine takes a few minutes to start and copy its files, and that time is billed too |
| steps need **different sizes**: each task gets the machine it needs | every task copies the **same big files** (one ACAF chromosome took about an hour just to copy) |
| you use **spot** machines (`--preemptible`): much cheaper, and dsub restarts a task if Google stops it | a **mistake** runs in every task: 22 tasks, 22 times the cost. That's why we test in the app first |
| you **stop the app** while the jobs run | |

In this class, both ways cost about $1. The workflow's gain is **time**: about 20 minutes instead of several hours.

### The words

| Word | Means |
|---|---|
| **Job** | one dsub command. It can have many **tasks**: today, one task per chromosome |
| **Image** (container) | a packaged computer system with your software already installed (here, PLINK 2). Each task runs inside one |
| **Input** | a file the task needs. dsub **copies** it from a bucket onto the task's machine first (*localizing*) |
| **Output** | a file the task makes. dsub copies it **back** to your bucket at the end (*delocalizing*) |
| **Script** | the commands each task runs: today, `gwas_one_chr.sh` |
| **Logging** | where dsub saves each task's log file. Your first stop when something fails |
| **Google Batch** | the Google Cloud service that creates, runs, and deletes the machines |

### How a dsub job runs

```
your app ── dsub ──► Google Batch ──► machine for chr1  ─┐
                                 ├──► machine for chr2  ─┤  each one: copy inputs in,
                                 ├──► ...                 ├─ run the script in the image,
                                 └──► machine for chr22 ─┘  copy outputs back, shut down
                                                           │
your bucket ◄──────────────── results and logs ───────────┘
```

### Why the settings look complicated

*All of Us* workspaces sit inside a **security perimeter**: job machines have **no internet access**. That has two consequences:

- Every job uses the workspace's private **network** and **subnetwork**, with **no public address** (`--use-private-address`).
- **Images** can't come straight from Docker Hub. *All of Us* keeps a **mirror** of Docker Hub inside Google Cloud: its address goes in front of the image name.

dsub also uses a small **helper image** (Google's Cloud SDK) to copy files. Its default version is a **moving** tag. In **November 2026**, a new version of that image broke every dsub job (they failed with *exit status 127*). The fix was to **pin** the previous version. That's a lesson for your own work too: **pin the versions** of the images you use, so they can't change under you.

**💬 Discuss:** which steps of an analysis you know would you move to a workflow, and which would you keep in the app?

## Part 2. Setup: settings, files, and the bucket

### Settings and a helper function

The first cell loads our usual settings. The second sets the two things every dsub job in *All of Us* needs (the pinned helper image, and the *All of Us* mirror), then defines `aou_dsub()`: it adds the **standard** settings (network, region, service account, logging) to whatever you give it, prints the full command, and runs it.

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions
system("dsub --version")

In [ ]:
# Settings every dsub job in All of Us needs (tested November 2026)
Sys.setenv(
  # dsub's helper image, pinned: version 592 (November 2026) breaks dsub jobs
  DSUB_CLOUD_SDK_IMAGE = "gcr.io/google.com/cloudsdktool/google-cloud-cli:591.0.0-slim",
  # the All of Us mirror of Docker Hub (job machines can't reach Docker Hub itself)
  ARTIFACT_REGISTRY_DOCKER_REPO = "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)
MIRROR      <- Sys.getenv("ARTIFACT_REGISTRY_DOCKER_REPO")
PLINK_IMAGE <- paste0(MIRROR, "/biocontainer/plink2:alpha2.3_jan2020")   # PLINK 2, a pinned version

# Run dsub with the settings All of Us needs, plus your own arguments (...).
# Returns the job ID (the last line dsub prints).
aou_dsub <- function(..., wait = FALSE) {
  args <- c("--provider", "google-batch",
            "--project", BILLING_PROJECT, "--user-project", BILLING_PROJECT,
            "--regions", "us-central1",
            "--network", "global/networks/network",
            "--subnetwork", "regions/us-central1/subnetworks/subnetwork",
            "--use-private-address",
            "--service-account", SERVICE_ACCOUNT,
            "--logging", paste0(WORK_GS, "/logs"),
            ...,
            if (wait) "--wait")
  message("dsub ", paste(args, collapse = " "), "\n")
  out <- suppressWarnings(system2("dsub", shQuote(args), stdout = TRUE, stderr = TRUE))
  cat(out, sep = "\n")
  if (!is.null(attr(out, "status"))) stop("dsub reported an error: read the lines above.", call. = FALSE)
  invisible(tail(out, 1))
}

`system2()` runs a command-line program from R; `shQuote()` wraps each argument in quotes, so the shell passes it on unchanged. `...` passes on whatever arguments you give `aou_dsub()`.

### Your bucket's address, and your account

Inside the app, your bucket is a **folder** (`BUCKET_DIR`). Job machines don't see that folder: they need the bucket's **cloud address**, which starts with `gs://`. The Workbench command-line tool, `wb`, can look it up.

In [ ]:
SERVICE_ACCOUNT <- system("gcloud config get-value account", intern = TRUE)

# The bucket's gs:// address, from the Workbench CLI. If that fails, we assume the
# bucket has the same name as its folder (true for many workspaces).
BUCKET_GS <- local({
  name <- basename(BUCKET_NAME)
  found <- suppressWarnings(system2("wb", c("resource", "resolve", "--name", shQuote(name)),
                                    stdout = TRUE, stderr = FALSE))
  found <- str_subset(found, "^gs://")
  if (length(found) == 0) found <- paste0("gs://", name)
  str_remove(found[1], "/$")
})

WORK_DIR <- file.path(DATA_DIR, "workflows")      # the folder, as the app sees it
WORK_GS  <- paste0(BUCKET_GS, "/data/workflows")  # the same folder, as job machines see it
dir.create(WORK_DIR, showWarnings = FALSE)

BUCKET_GS

**🔍 Read the output:** an address like `gs://…`. We check it in a moment: if it's wrong, the check fails, and you can type the address in by hand (it's on the **Resources** tab: click your bucket).

### The GWAS files from 4.2

Each job needs three small files: the **people** to keep, the **phenotype** (case or control), and the **covariates**. If you ran **4.2**, they're already in your bucket; if not, this cell makes them the same way (from the sample saved in **4.1**, or a new one).

In [ ]:
GENO_DIR   <- file.path(DATA_DIR, "genomics")
keep_file  <- file.path(GENO_DIR, "keep.txt")
pheno_file <- file.path(GENO_DIR, "pheno.tsv")
covar_file <- file.path(GENO_DIR, "covar.tsv")

if (!all(file.exists(c(keep_file, pheno_file, covar_file)))) {
  dir.create(GENO_DIR, showWarnings = FALSE)
  if (file.exists(file.path(DATA_DIR, "gwas_samples.csv"))) {
    gwas_samples <- load_data("gwas_samples.csv", col_types = cols(person_id = "c"))
  } else {
    gwas_samples <- t2d_gwas_samples()   # runs the cohort query
    save_data(gwas_samples, "gwas_samples.csv")
  }
  write_lines(gwas_samples$person_id, keep_file)
  write_tsv(gwas_samples %>% transmute(`#IID` = person_id, t2d = if_else(cohort == "case", 2, 1)),
            pheno_file)
  write_tsv(gwas_samples %>% transmute(`#IID` = person_id, age,
                                       sex_male = if_else(sex_at_birth == "Male", 1, 0),
                                       across(PC1:PC10)),
            covar_file)
  message("Made the GWAS files")
}

# The same three files, as job machines see them
KEEP_GS  <- paste0(BUCKET_GS, "/data/genomics/keep.txt")
PHENO_GS <- paste0(BUCKET_GS, "/data/genomics/pheno.tsv")
COVAR_GS <- paste0(BUCKET_GS, "/data/genomics/covar.tsv")

# Check the address: this lists the three files in the bucket (names only)
system2("gcloud", c("storage", "ls", paste0(BUCKET_GS, "/data/genomics/")), stdout = TRUE) %>%
  str_subset("keep.txt|pheno.tsv|covar.tsv")

**🔍 Read the output:** the three `gs://…` file names. If you see an error instead, the bucket address is wrong: set it by hand, e.g. `BUCKET_GS <- "gs://my-bucket-name"`, then rerun the line that sets `WORK_GS` and this cell.

### The ClinVar PLINK files

The *All of Us* genomes come in several **callsets** (see *Where the omics data lives*). We use **ClinVar**: only the variants listed in the ClinVar database, so the files are **small** enough to copy in minutes. They come in PLINK 1 format (`.bed`, `.bim`, `.fam`), one set per chromosome.

In [ ]:
CLINVAR_GS  <- "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/clinvar/plink_bed"
CLINVAR_DIR <- file.path(aou_data_dir("v9"), "wgs", "short_read", "snpindel", "clinvar", "plink_bed")

clinvar_files <- tibble(file = list.files(CLINVAR_DIR, pattern = "\\.bed$")) %>%
  mutate(chr = as.integer(str_match(file, "chr(\\d+)\\.bed$")[, 2]),
         size_gb = round(file.size(file.path(CLINVAR_DIR, file)) / 1e9, 2)) %>%
  filter(chr %in% 1:22) %>%
  arrange(chr)

clinvar_files
sum(clinvar_files$size_gb)

**🔍 Read the output:** one `.bed` file per chromosome, 1 to 22, and their sizes in GB. Each task needs a **disk** big enough for its inputs and outputs, so we set the disk size from the biggest file (next cell). For comparison, the **ACAF** files are many times bigger (Part 7).

In [ ]:
DISK_GB <- ceiling(max(clinvar_files$size_gb) * 1.5 + 10)   # room for the inputs, outputs, and spare
DISK_GB

## Part 3. Test locally: one chromosome in the app

**Always test on something small first.** A mistake in a 22-job workflow costs 22 times as much, and takes longer to find.

The script `gwas_one_chr.sh` (next to this notebook) is the GWAS from **4.2**: QC filters, then `plink2 --glm`. It reads every file name from an **environment variable** (`CHR`, `BED`, `KEEP`, …), so the **same script** can run in the app and in a job: only the variables change. Let's read it.

In [ ]:
cat(read_lines("gwas_one_chr.sh"), sep = "\n")

| Line | Does |
|---|---|
| `set -euo pipefail` | stop at the first error. Without it, a failed step could still look like a successful job |
| `mkdir -p "${OUT_DIR}"` | make the output folder, if it isn't there |
| `--bfile "${BED%.bed}"` | the PLINK files: `${BED%.bed}` is the `.bed` file name without `.bed` |
| `--keep`, `--maf`, `--geno`, `--hwe` | our sample, and the QC filters from **4.2** |
| `--pheno`, `--covar`, `--glm …` | the association test from **4.2** |
| `--out "${OUT_DIR}/chr${CHR}"` | results named after the chromosome, e.g. `chr21.t2d.glm.logistic.hybrid` |

Now run it **in the app**, on chromosome **21** (one of the smallest), reading the files from the mounted folders. It takes a minute or two.

In [ ]:
LOCAL_OUT <- file.path(tempdir(), "local_test")   # on the app's disk, not in the bucket

local_run <- system2("bash", "gwas_one_chr.sh", stdout = TRUE, stderr = TRUE, env = c(
  "CHR=21",
  paste0("BED=",     shQuote(file.path(CLINVAR_DIR, "clinvar.chr21.bed"))),
  paste0("KEEP=",    shQuote(keep_file)),
  paste0("PHENO=",   shQuote(pheno_file)),
  paste0("COVAR=",   shQuote(covar_file)),
  paste0("OUT_DIR=", shQuote(LOCAL_OUT))
))

if (!is.null(attr(local_run, "status"))) cat(tail(local_run, 20), sep = "\n")  # if it failed: why

# The key lines from PLINK's log
str_subset(read_lines(file.path(LOCAL_OUT, "chr21.log")), "samples|variants|written")

**🔍 Read the output:** PLINK's summary: how many people it kept (5,000, or fewer if some aren't in the ClinVar files), how many variants the QC filters removed and kept, and the name of the results file. Most ClinVar variants are **rare**, so `--maf 0.01` removes most of them.

If something went wrong (a typo, a missing file), you found out in a minute, for free. Fix it here, **before** you launch any jobs.

In [ ]:
# Read a plink2 --glm results file. chr: the chromosome as a number (with or without "chr")
read_glm <- function(file) {
  read_tsv(file, col_types = cols(.default = "c")) %>%
    mutate(chr = as.integer(str_remove(`#CHROM`, "^chr")),
           across(any_of(c("POS", "OBS_CT", "OR", "P")), as.numeric))
}

local_results <- read_glm(list.files(LOCAL_OUT, pattern = "glm", full.names = TRUE)[1])
nrow(local_results)
local_results %>% arrange(P) %>% select(ID, A1, OBS_CT, OR, P) %>% head(5)

**🔍 Read the output:** the number of variants tested on chromosome 21, and the five with the smallest p-values. These are **summary statistics** (no one's individual data), so they're safe to look at. We'll compare them with the job's results in Part 4.

## Part 4. One dsub job: the same chromosome in the cloud

Now the **same script** runs as a dsub job. Each argument answers a question:

| Argument | Answers |
|---|---|
| `--image` | Which software? PLINK 2, from the *All of Us* mirror |
| `--script` | What should it run? `gwas_one_chr.sh` |
| `--env CHR=21` | A setting (not a file) for the script |
| `--input BED=gs://…` | Which files to copy **onto** the machine. The script sees `BED` as the copy's path on the machine |
| `--output-recursive OUT_DIR=gs://…` | Which folder to copy **back** to the bucket at the end |
| `--machine-type`, `--disk-size` | How big a machine and disk |
| `--name` | A short name for the job (it starts the job ID) |
| `--wait` | Wait until the job finishes (the cell keeps running until then) |

⚠️ The `.bim` and `.fam` files are **inputs too**, though the script never names them: PLINK needs them next to the `.bed` file. **Every file a job needs must be an input**: the machine only has what you give it.

This takes about **10 minutes**: a few to start the machine and copy the files, and a few for PLINK. While you wait, read on to Part 5.

In [ ]:
job_chr21 <- aou_dsub(
  "--name", "gwas-chr21",
  "--image", PLINK_IMAGE,
  "--machine-type", "n2-standard-2",
  "--disk-size", DISK_GB,
  "--script", "gwas_one_chr.sh",
  "--env", "CHR=21",
  "--input", paste0("BED=", CLINVAR_GS, "/clinvar.chr21.bed"),
  "--input", paste0("BIM=", CLINVAR_GS, "/clinvar.chr21.bim"),
  "--input", paste0("FAM=", CLINVAR_GS, "/clinvar.chr21.fam"),
  "--input", paste0("KEEP=", KEEP_GS),
  "--input", paste0("PHENO=", PHENO_GS),
  "--input", paste0("COVAR=", COVAR_GS),
  "--output-recursive", paste0("OUT_DIR=", WORK_GS, "/test_chr21"),
  wait = TRUE
)
job_chr21

**🔍 Read the output:** first the full `dsub` command (from `aou_dsub()`); then dsub's messages: the **job ID**, the `dstat` and `ddel` commands for this job, and, at the end, `SUCCESS` (or `FAILURE`: see Part 7). The last line is the job ID, which `aou_dsub()` returns.

### Did the job give the same results?

The job copied its results to `data/workflows/test_chr21/` in your bucket. We copy them onto the app's disk with `gcloud storage cp` (the mounted folder can take a minute to show new files) and compare them with the local test.

In [ ]:
JOB_OUT <- file.path(tempdir(), "job_test")
dir.create(JOB_OUT, showWarnings = FALSE)
system2("gcloud", c("storage", "cp", paste0(WORK_GS, "/test_chr21/*"), JOB_OUT))

job_results <- read_glm(list.files(JOB_OUT, pattern = "glm", full.names = TRUE)[1])

compare <- inner_join(local_results, job_results, by = c("ID", "A1"), suffix = c("_app", "_job"))
c(variants_app = nrow(local_results), variants_job = nrow(job_results), matched = nrow(compare))
max(abs(log10(compare$P_app) - log10(compare$P_job)), na.rm = TRUE)

**🔍 Read the output:**

- The same number of variants in both, all matched.
- The last number is the **biggest difference** between the two runs' p-values, on the log scale. Close to **0** means the same results. Tiny differences can come from the **PLINK version**: the job's image has a 2020 version of PLINK 2, the app a newer one. Another reason to pin versions, and to report the version you used.

### The job's log

dsub saved a log for the job in `data/workflows/logs/`. It has dsub's own messages (copying files, starting the script) and everything the script printed. When a job fails, **read the log first**.

In [ ]:
system2("gcloud", c("storage", "ls", paste0(WORK_GS, "/logs/")), stdout = TRUE) %>% str_subset(job_chr21)

In [ ]:
job_log <- system2("gcloud", c("storage", "cat", paste0(WORK_GS, "/logs/", job_chr21, ".log")),
                   stdout = TRUE)
str_subset(job_log, "Localizing|Delocalizing|PLINK v|samples|variants") %>% head(15)

**🔍 Read the output:** three log files per job: `….log` (everything), `…-stdout.log` (what the script printed), and `…-stderr.log` (its errors and warnings). The lines we picked show the steps in order: dsub **copying the inputs in**, PLINK's version and its summary, and dsub **copying the outputs back**.

## Part 5. All 22 chromosomes at once

One job worked. To run **many**, we don't call dsub 22 times: we give it a **tasks file**, a table with **one row per task** and **one column per setting**. Each column's heading says what it is (`--env`, `--input`, `--output-recursive`) and its name. dsub starts one task per row, all at the same time.

Settings that are the **same** for every task (the image, the script, our three small files) stay in the dsub command.

In [ ]:
tasks <- tibble(
  `--env CHR`                = clinvar_files$chr,
  `--input BED`              = paste0(CLINVAR_GS, "/clinvar.chr", clinvar_files$chr, ".bed"),
  `--input BIM`              = paste0(CLINVAR_GS, "/clinvar.chr", clinvar_files$chr, ".bim"),
  `--input FAM`              = paste0(CLINVAR_GS, "/clinvar.chr", clinvar_files$chr, ".fam"),
  `--output-recursive OUT_DIR` = paste0(WORK_GS, "/gwas_clinvar/chr", clinvar_files$chr)
)

tasks_file <- file.path(WORK_DIR, "gwas_tasks.tsv")
write_tsv(tasks, tasks_file)
nrow(tasks)
tasks %>% select(`--env CHR`, `--input BED`) %>% head(3)

**🔍 Read the output:** 22 tasks, one per chromosome, and the first three rows. These are file **names**, not data, so they're safe to look at.

This time we **don't wait**: the cell launches the job and finishes, and the 22 tasks run in the background. You could even stop the app now: the jobs keep running, and the results will be in your bucket.

In [ ]:
job_all <- aou_dsub(
  "--name", "gwas-clinvar",
  "--image", PLINK_IMAGE,
  "--machine-type", "n2-standard-2",
  "--disk-size", DISK_GB,
  "--script", "gwas_one_chr.sh",
  "--input", paste0("KEEP=", KEEP_GS),
  "--input", paste0("PHENO=", PHENO_GS),
  "--input", paste0("COVAR=", COVAR_GS),
  "--tasks", tasks_file
)
job_all

### Check on the jobs: `dstat`

`dstat` shows the status of a job's tasks. `--summary` counts tasks by status: **RUNNING**, **SUCCESS**, **FAILURE**. Run this cell a few times while you wait.

In [ ]:
# Show (and return) the status of a job's tasks. Extra arguments go to dstat.
dstat <- function(job_id, ..., show = TRUE) {
  out <- system2("dstat", c("--provider", "google-batch", "--project", BILLING_PROJECT,
                            "--location", "us-central1", "--jobs", shQuote(job_id),
                            "--status", "'*'", ...),
                 stdout = TRUE, stderr = show)
  if (show) cat(out, sep = "\n")
  invisible(out)
}

dstat(job_all, "--summary")

**🔍 Read the output:** a small table: the job, and how many of its tasks are in each status. At first they're all **RUNNING** (that includes starting the machine); after 10 to 15 minutes they should all be **SUCCESS**.

When you're ready to wait for the rest, run the next cell: `--wait` makes `dstat` return only when every task has finished.

In [ ]:
dstat(job_all, "--wait", show = FALSE)
dstat(job_all, "--summary")

### How long did each task take?

With `--full --format json`, `dstat` reports every detail of every task, including when it was **created**, when it **started**, and when it **ended**.

In [ ]:
details <- jsonlite::fromJSON(paste(dstat(job_all, "--full", "--format", "json", show = FALSE),
                                     collapse = "\n"))

timing <- tibble(
  chr     = as.integer(details$envs$CHR),
  status  = details$status,
  minutes = round(as.numeric(difftime(ymd_hms(details$`end-time`),
                                      ymd_hms(details$`create-time`), units = "mins")), 1)
) %>% arrange(chr)

timing
c(longest_task = max(timing$minutes),
  all_tasks_one_after_another = sum(timing$minutes))

**🔍 Read the output:** one row per chromosome: its status and how many minutes it took from launch to finish (bigger chromosomes take longer). The last two numbers are the point of the class: the whole job took about as long as its **longest** task, not the **sum** of all of them.

## Part 6. Combine the results

Each task wrote its results to its own folder in `data/workflows/gwas_clinvar/`. We copy the result files onto the app's disk and stack them into one table.

In [ ]:
ALL_OUT <- file.path(tempdir(), "gwas_clinvar")
dir.create(ALL_OUT, showWarnings = FALSE)
system2("gcloud", c("storage", "cp", paste0(WORK_GS, "/gwas_clinvar/*/chr*.glm.*"), ALL_OUT))

result_files <- list.files(ALL_OUT, pattern = "glm", full.names = TRUE)
length(result_files)   # one per chromosome

gwas_all <- map(result_files, read_glm) %>% bind_rows() %>% arrange(chr, POS)
nrow(gwas_all)
gwas_all %>% count(chr, name = "variants_tested")

**🔍 Read the output:** 22 files; the total number of variants tested; and how many on each chromosome. These are counts of **variants**, not people, so small numbers are fine to share.

### A Manhattan plot

A **Manhattan plot** shows every variant's p-value along the genome, one colour band per chromosome (as in **4.2** and **4.3**). The dashed line is the genome-wide significance threshold, 5 × 10⁻⁸.

In [ ]:
plot_data <- gwas_all %>%
  filter(!is.na(P)) %>%
  mutate(x    = row_number(),          # variants in order, chromosome by chromosome
         band = factor(chr %% 2))      # alternate colours by chromosome

axis_labels <- plot_data %>% group_by(chr) %>% summarise(mid = median(x))

ggplot(plot_data, aes(x, -log10(P), colour = band)) +
  geom_point(size = 0.8) +
  geom_hline(yintercept = -log10(5e-8), linetype = "dashed") +
  scale_x_continuous(breaks = axis_labels$mid, labels = axis_labels$chr) +
  scale_colour_manual(values = c("#08519c", "#6baed6"), guide = "none") +
  theme_bw() +
  labs(x = "Chromosome (variants in order)", y = "-log10(p-value)",
       title = "Type 2 diabetes: GWAS of the common ClinVar variants")

**🔍 Read the output:** each dot is one variant. Points above the dashed line would be genome-wide significant. Expect **few or none**: after the QC filters, only the **common** ClinVar variants are left, and with 5,000 people, only strong effects reach that line. (The x-axis is the variants' **order**, not their position, because ClinVar variants are spread unevenly.)

In [ ]:
gwas_all %>% arrange(P) %>% select(chr, POS, ID, A1, OBS_CT, OR, P) %>% head(10)
save_data(gwas_all, "gwas_clinvar_all.csv")

**🔍 Read the output:** the 10 variants with the smallest p-values, with their odds ratios per copy of `A1`. These summary statistics are saved as `gwas_clinvar_all.csv` in your bucket.

**💬 Discuss:** is a ClinVar variant with a small p-value here a variant that **causes** diabetes? (Think about what ClinVar lists, and about the region around *TCF7L2* in 4.2.)

## Part 7. Monitor, troubleshoot, scale up, and clean up

### Commands for managing jobs

| Task | Command (or our R function) |
|---|---|
| See a job's status | `dstat(job_id)`, or `dstat(job_id, "--summary")` (our R function) |
| See every detail | `dstat(job_id, "--full")` |
| Wait for a job to finish | `dstat(job_id, "--wait")` |
| Cancel a job (all its tasks) | `ddel --provider google-batch --project … --location us-central1 --jobs JOB_ID` |
| Rerun some tasks | `--tasks gwas_tasks.tsv 5-7` runs only rows 5 to 7 |
| Read a log | `gcloud storage cat gs://…/logs/JOB_ID.log` (or `….1.log` for task 1) |

### When a job fails

| Symptom (in `dstat` or the log) | Likely cause | Fix |
|---|---|---|
| `FAILURE` with **exit status 127**, before your script runs | dsub's helper image broke or can't be reached | pin `DSUB_CLOUD_SDK_IMAGE` to a version that works (Part 2) |
| An error about the **image**, or a job that never starts | the image isn't from a Google registry | put the *All of Us* mirror in front of Docker Hub images (`MIRROR`) |
| `No such file or directory` in the log | a file the script uses wasn't an **input** | add it with `--input` (e.g. the `.bim` and `.fam` files) |
| `No space left on device` | the disk is too small for the inputs and outputs | a bigger `--disk-size` |
| PLINK runs out of memory, or a task is killed | the machine is too small | a bigger `--machine-type` (e.g. `n2-highmem-4`) |
| `401 … Requester pays bucket access requires authentication`, then a retry | a short delay while the machine signs in | nothing: dsub retries, and the job continues |
| `Error: No variants remaining` | the filters removed every variant on that chromosome | check the filters; or leave that chromosome out |
| Tasks stay in the queue for a long time | the project's limit on machines or CPUs | wait, or run fewer tasks at a time |

### Scaling up: from ClinVar to ACAF

For a real GWAS, use the **ACAF** files (every variant common in at least one ancestry group). The steps are the same; the files are much **bigger**. In the *All of Us* tutorial, one ACAF chromosome (21) took about **60 minutes** to copy onto the machine and about **28 minutes** for PLINK, on a machine with 32 CPUs (`n2-highmem-32`) and a 400 GB disk.

| | ClinVar (today) | ACAF |
|---|---|---|
| **Variants** | those listed in ClinVar | all variants common in at least one ancestry group |
| **File sizes** | GB per chromosome | tens to hundreds of GB per chromosome |
| **Machine** | `n2-standard-2` | `n2-highmem-16` to `-32` |
| **Disk** | under 100 GB | 400 GB or more |
| **Time per task** | 10 to 15 minutes | 1 to 2+ hours |
| **Cost** | under $1 in all | tens of dollars or more: **test on one chromosome first** |

Ways to save time and money:

- **Spot (preemptible) machines:** `--preemptible` plus `--retries 2`. They cost much less, but Google can stop them at any time; dsub starts them again.
- **Split chromosomes into regions** (`--chr 1 --from-bp … --to-bp …` in PLINK): more, smaller tasks.
- **Read only what you need:** the `.pgen` files, with `--keep` and regions, as in **4.2**.

### Clean up

Job results stay in your bucket until you delete them, and bucket storage costs a little every month. When you're done, delete the folders you don't need. For example, the chromosome 21 test:

In [ ]:
# Delete the chromosome 21 test results (uncomment to run)
# system2("gcloud", c("storage", "rm", "--recursive", paste0(WORK_GS, "/test_chr21")))

## Part 8. Cromwell and Nextflow

dsub is the **simplest** workflow tool: one command, one script, a table of tasks. It's great for **one step on many files**. When an analysis has **many steps** (e.g. QC → GWAS → plots), where each step uses the outputs of the one before, a **workflow language** is easier: you describe the steps and how they connect, and the tool runs them in the right order, in parallel where it can, and can **resume** after a failure.

| | dsub | Cromwell | Nextflow |
|---|---|---|---|
| **You write** | a shell script and a dsub command | a **WDL** file (*Workflow Description Language*) and a JSON file of inputs | a **Nextflow** (`.nf`) file |
| **Best for** | one step on many files | multi-step pipelines; many published genomics workflows (e.g. GATK) | multi-step pipelines; the **nf-core** community pipelines |
| **Run with** | `dsub …` | `java -jar $CROMWELL_JAR run …`, or the **Workflows** page in the Workbench | `nextflow run …` |
| **Shared workflows** | | [Dockstore](https://dockstore.org/) | [nf-core](https://nf-co.re/), [Dockstore](https://dockstore.org/) |
| **Learning curve** | low | medium | medium |

All three run their jobs on **Google Batch**, so the *All of Us* rules from Part 1 apply to each: the private network, no public addresses, and images from the *All of Us* mirror or other Google registries. Each needs a short **configuration file** with these settings.

👉 **To start:** the *All of Us* tutorial workspace has step-by-step notebooks for Cromwell and Nextflow, including the configuration files. The companion page for this class shows their key parts.

## Exercises

**1.** Run only chromosomes 19 to 22, in a new job. (Hint: the tasks file has a header row, so chromosome 1 is task 1.)

<details><summary>Answer</summary>

Add the task numbers after the tasks file:

```r
job_some <- aou_dsub(
  "--name", "gwas-clinvar-19-22",
  "--image", PLINK_IMAGE, "--machine-type", "n2-standard-2", "--disk-size", DISK_GB,
  "--script", "gwas_one_chr.sh",
  "--input", paste0("KEEP=", KEEP_GS),
  "--input", paste0("PHENO=", PHENO_GS),
  "--input", paste0("COVAR=", COVAR_GS),
  "--tasks", tasks_file, "19-22"
)
```

Rows 19 to 22 are chromosomes 19 to 22, because `clinvar_files` is sorted by chromosome. They'll overwrite the earlier results for those chromosomes (same output folders), with the same results.
</details>

**2.** Make the job cheaper with **spot** machines. Which two arguments would you add, and what's the risk?

<details><summary>Answer</summary>

`"--preemptible", "--retries", "2"`. Spot machines cost much less, but Google can stop one at any time. `--retries 2` tells dsub to start a stopped task again, up to 2 times. The risk: a task that keeps getting stopped takes longer, and long tasks are stopped more often. Spot machines suit short tasks like ours.
</details>

**3.** A classmate's job fails. The log says `plink2: Error: Failed to open …/clinvar.chr7.bim`. What's wrong?

<details><summary>Answer</summary>

The `.bim` file wasn't an **input**: the machine only has the files you give it with `--input`. PLINK needs the `.bed`, `.bim`, and `.fam` files together, so all three must be inputs (as in our tasks file).
</details>

**4.** Why did we test in the app before launching any jobs?

<details><summary>Answer</summary>

A mistake shows up in a minute, for free, and is easy to fix in the app. In a 22-task job, the same mistake costs 22 machines, takes longer to show up (each machine must start and copy its files first), and is harder to read (in a log, not on screen).
</details>

## Common errors

| Error | Cause | Fix |
|---|---|---|
| `dsub: command not found` | dsub isn't in this app | use an *All of Us* Jupyter app (dsub is installed there) |
| `dsub reported an error` from `aou_dsub()`, with `FAILURE` above | a task failed | read the job's log (Part 4), then see *When a job fails* (Part 7) |
| `gcloud storage ls` fails in Part 2 | the bucket address is wrong | set `BUCKET_GS` by hand, from the **Resources** tab |
| `Blank line(s) found in …gwas_tasks.tsv` | an empty row in the tasks file | rewrite it with `write_tsv()` (as above) |
| `Unexpected number of fields` | a row of the tasks file has a missing or extra value | check the tasks table for `NA`s |
| No result files in Part 6 | some tasks haven't finished, or failed | `dstat(job_all, "--summary")`, then read the logs |

## Key takeaways

- Use a **workflow tool** when the **same step** has to run on **many** or **big** files: each task gets its own machine, and they run **in parallel**
- A dsub job is an **image**, a **script**, **inputs** (copied onto the machine), **outputs** (copied back), and a **log**
- **Test small first:** the script in the app, then one job, then all of them
- **Every file a task needs must be an input**: the machine has nothing else
- In *All of Us*: the private network, no public address, images from the **mirror**, and **pinned** versions
- `dstat` to monitor, `ddel` to cancel, and the **log** to find out why a task failed
- For **multi-step** pipelines, use a workflow language: **WDL** with Cromwell, or **Nextflow**

> ⚠️ **Before you share or commit this notebook**, clear the outputs: **Edit → Clear Outputs of All Cells**. dsub's messages include your workspace's project ID and your bucket's address.